In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder

# Load your data (example with ImageDataGenerator)
data_dir = '/content/base_dir'  # Specify your dataset path
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

# Create training and validation generators
train_generator = datagen.flow_from_directory(data_dir,
                                              target_size=(224, 224),
                                              batch_size=32,
                                              class_mode='categorical',
                                              subset='training',
                                              shuffle=False)

val_generator = datagen.flow_from_directory(data_dir,
                                            target_size=(224, 224),
                                            batch_size=32,
                                            class_mode='categorical',
                                            subset='validation',
                                            shuffle=False)

# Load two pre-trained models from files
model1 = tf.keras.models.load_model('/content/drive/MyDrive/_FYP_ResNet50_best_model (2).h5')
model2 = tf.keras.models.load_model('/content/drive/MyDrive/vit_skin_cancer_classifier.h5')

# Display summaries of both base models
print("Model 1 Summary:")
model1.summary()
print("\nModel 2 Summary:")
model2.summary()

# Generate predictions from both models on the training and validation sets
train_preds_model1 = model1.predict(train_generator)
train_preds_model2 = model2.predict(train_generator)

val_preds_model1 = model1.predict(val_generator)
val_preds_model2 = model2.predict(val_generator)

# Concatenate the predictions from both models to form the training data for the meta-learner
X_train_meta = np.hstack([train_preds_model1, train_preds_model2])
X_val_meta = np.hstack([val_preds_model1, val_preds_model2])

# Get the true labels for the meta-learner
y_train = train_generator.classes
y_val = val_generator.classes

# One-hot encoding of the labels for neural network training
# Change 'sparse=False' to 'sparse_output=False'
encoder = OneHotEncoder(sparse_output=False)  # Updated line
y_train_onehot = encoder.fit_transform(y_train.reshape(-1, 1))
y_val_onehot = encoder.transform(y_val.reshape(-1, 1))

# Create a complex meta-learner model (deep neural network)
meta_model = models.Sequential([
    layers.InputLayer(input_shape=(X_train_meta.shape[1],)),  # Number of input features from base models
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(train_generator.num_classes, activation='softmax')  # Number of classes in your dataset
])

# Display the summary of the meta-learner model
print("\nMeta-Learner Model Summary:")
meta_model.summary()

# Compile the meta-learner
meta_model.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

# Train the meta-learner using the concatenated predictions from the base models
meta_model.fit(X_train_meta, y_train_onehot, epochs=50, batch_size=32, validation_data=(X_val_meta, y_val_onehot))

# Evaluate the performance of the meta-model on the validation set
val_loss, val_accuracy = meta_model.evaluate(X_val_meta, y_val_onehot)
print(f"Validation Accuracy: {val_accuracy:.2f}")

# Make predictions on the validation set
val_predictions = meta_model.predict(X_val_meta)
val_pred_classes = np.argmax(val_predictions, axis=1)

# Print classification report for validation set
print(classification_report(y_val, val_pred_classes))

# Save the meta-learner model to disk
meta_model.save('meta_learner_model.h5')
print("Meta-learner model saved as 'meta_learner_model.h5'.")

# Later, load the meta-learner model from disk
loaded_meta_model = tf.keras.models.load_model('meta_learner_model.h5')

# Display the summary of the loaded meta-learner model
print("\nLoaded Meta-Learner Model Summary:")
loaded_meta_model.summary()

# Evaluate the loaded meta-model on the validation set to verify its performance
loaded_val_loss, loaded_val_accuracy = loaded_meta_model.evaluate(X_val_meta, y_val_onehot)
print(f"Validation Accuracy (Loaded Model): {loaded_val_accuracy:.2f}")


Found 31607 images belonging to 2 classes.
Found 7900 images belonging to 2 classes.


Model 1 Summary:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 224, 224, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_pad (ZeroPadding2D) │ (None, 230, 230, 3)    │              0 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_conv (Conv2D)       │ (None, 112, 112, 64)   │          9,472 │ conv1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_bn                  │ (None, 112, 112, 64)   │            256 │ conv1_conv[0][0]       │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_relu (Activation)   │ (None, 112, 112, 64)   │              0 │ conv1_bn[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pad (ZeroPadding2D) │ (None, 114, 114, 64)   │              0 │ conv1_relu[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pool (MaxPooling2D) │ (None, 56, 56, 64)     │              0 │ pool1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_conv       │ (None, 56, 56, 64)     │          4,160 │ pool1_pool[0][0]       │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_bn         │ (None, 56, 56, 64)     │            256 │ conv2_block1_1_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_relu       │ (None, 56, 56, 64)     │              0 │ conv2_block1_1_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_conv       │ (None, 56, 56, 64)     │         36,928 │ conv2_block1_1_relu[0… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_bn         │ (None, 56, 56, 64)     │            256 │ conv2_block1_2_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_relu       │ (None, 56, 56, 64)     │              0 │ conv2_block1_2_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_0_conv       │ (None, 56, 56, 256)    │         16,640 │ pool1_pool[0][0]       │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_3_conv       │ (None, 56, 56, 256)    │         16,640 │ conv2_block1_2_relu[0… │
│ (Conv2D)                  │                        │                │                        │
├──────────────────────

 Total params: 26,211,716 (99.99 MB)

...

 Total params: 44,356 (173.27 KB)

 Trainable params: 43,458 (169.76 KB)

 Non-trainable params: 896 (3.50 KB)

 Optimizer params: 2 (12.00 B)

247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9996 - loss: 0.0601
Validation Accuracy (Loaded Model): 0.98


In [3]:
from google.colab import files

files.upload() #this will prompt you to upload the kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"malaikahasnatmalik","key":"0875200a145725f6f909b13d8d19a0cf"}'}

In [4]:
!ls -lha kaggle.json

-rw-r--r-- 1 root root 74 Oct 10 17:30 kaggle.json


In [5]:
!pip install -q kaggle

In [6]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [7]:
!chmod 600 /root/.kaggle/kaggle.json

In [8]:

!pwd

/content


In [9]:
!kaggle datasets list

ref                                                           title                                                size  lastUpdated          downloadCount  voteCount  usabilityRating  
------------------------------------------------------------  --------------------------------------------------  -----  -------------------  -------------  ---------  ---------------  
lainguyn123/student-performance-factors                       Student Performance Factors                          94KB  2024-09-02 10:53:57          28126        496  1.0              
abdulszz/spotify-most-streamed-songs                          Spotify Most Streamed Songs                          60KB  2024-09-07 18:23:14          10651        139  1.0              
valakhorasani/mobile-device-usage-and-user-behavior-dataset   Mobile Device Usage and User Behavior Dataset        11KB  2024-09-28 20:21:12           3404         66  1.0              
thomasfranois/worlds-best-restaurants                         World's 

In [10]:

!kaggle datasets download -d utkarshps/skin-cancer-mnist10000-ham-augmented-dataset

Dataset URL: https://www.kaggle.com/datasets/utkarshps/skin-cancer-mnist10000-ham-augmented-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.77G/2.77G [02:55<00:00, 20.8MB/s]
100% 2.77G/2.77G [02:55<00:00, 17.0MB/s]


In [11]:
!unzip skin-cancer-mnist10000-ham-augmented-dataset

Streaming output truncated to the last 5000 lines.
  inflating: base_dir/train_dir/vasc/_124_9541474.jpg  
  inflating: base_dir/train_dir/vasc/_124_9772208.jpg  
  inflating: base_dir/train_dir/vasc/_125_1175166.jpg  
  inflating: base_dir/train_dir/vasc/_125_1449418.jpg  
  inflating: base_dir/train_dir/vasc/_125_2119583.jpg  
  inflating: base_dir/train_dir/vasc/_125_2119666.jpg  
  inflating: base_dir/train_dir/vasc/_125_2164554.jpg  
  inflating: base_dir/train_dir/vasc/_125_2256805.jpg  
  inflating: base_dir/train_dir/vasc/_125_2488751.jpg  
  inflating: base_dir/train_dir/vasc/_125_3149868.jpg  
  inflating: base_dir/train_dir/vasc/_125_331766.jpg  
  inflating: base_dir/train_dir/vasc/_125_3347656.jpg  
  inflating: base_dir/train_dir/vasc/_125_3881906.jpg  
  inflating: base_dir/train_dir/vasc/_125_3983138.jpg  
  inflating: base_dir/train_dir/vasc/_125_4217599.jpg  
  inflating: base_dir/train_dir/vasc/_125_4769564.jpg  
  inflating: base_dir/train_dir/vasc/_125_4771093.jpg 